In [1]:
import sys
import optuna
import numpy as np


from shapely.geometry import Point

sys.path.append("../..")
from src.qubo_windfarm_layout.model import get_farm_area, compute_aep_from_coords
from src.qubo_windfarm_layout.evaluation import load_layout_coordinates, get_farm_area

/home/maicolnicolini/code/qubo-wind-farm-layout/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import numpy as np
import pandas as pd
import optuna

from scipy.spatial import cKDTree
from shapely.geometry import Point


def optuna_promising_turbines_refinement(
    layout_yaml,
    candidate_locations,
    wake_loss_matrix,
    step_m,
    radius_m,
    n_promising_turbines=15,
    n_trials_per_turbine=30,
    min_distance=396.0,
    max_outer_iterations=30,
    min_improvement=1e-6,
    match_tol=1e-6,
    seed=42,
    verbose=True,
):
    """
    Hybrid physics-based local refinement.

    Outer iteration:
        1. rank turbines using pairwise wake interaction;
        2. select the most promising turbines;
        3. for each selected turbine:
               optimize dx, dy with Optuna
               while keeping the other 80 turbines fixed;
        4. select the globally best move;
        5. accept ONE move;
        6. update pairwise interaction matrix;
        7. repeat.

    Optuna objective:
        maximize full AEP.
    """

    # ==================================================
    # INITIAL LAYOUT
    # ==================================================

    current_positions = np.asarray(
        load_layout_coordinates(layout_yaml),
        dtype=float,
    )

    candidate_locations = np.asarray(
        candidate_locations,
        dtype=float,
    )

    wake_loss_matrix = np.asarray(
        wake_loss_matrix,
        dtype=float,
    )

    n_turbines = len(current_positions)

    _, farm_area = get_farm_area()

    # --------------------------------------------------
    # Map initial YAML layout -> wake matrix indices
    # --------------------------------------------------

    candidate_tree = cKDTree(
        candidate_locations
    )

    distances, selected_indices = (
        candidate_tree.query(
            current_positions,
            k=1,
        )
    )

    if np.any(distances > match_tol):
        raise ValueError(
            "Il layout YAML non è compatibile con "
            "candidate_locations. "
            f"Max distance = {distances.max():.6f} m"
        )

    if len(np.unique(selected_indices)) != n_turbines:
        raise ValueError(
            "Più turbine sono state associate "
            "alla stessa candidate location."
        )

    # Interaction matrix only for current 81 turbines
    current_L = wake_loss_matrix[
        np.ix_(
            selected_indices,
            selected_indices,
        )
    ].copy()

    # Single-turbine AEP
    A0 = float(
        compute_aep_from_coords(
            [current_positions[0]]
        )
    )

    current_aep = float(
        compute_aep_from_coords(
            current_positions
        )
    )

    initial_aep = current_aep

    history = []
    all_trials = []

    # ==================================================
    # LOCAL GRID
    # ==================================================

    n_steps = int(
        np.floor(radius_m / step_m)
    )

    offsets = (
        np.arange(
            -n_steps,
            n_steps + 1,
        )
        * step_m
    )

    dx_grid, dy_grid = np.meshgrid(
        offsets,
        offsets,
    )

    local_offsets = np.column_stack(
        (
            dx_grid.ravel(),
            dy_grid.ravel(),
        )
    )

    # circular neighbourhood
    local_offsets = local_offsets[
        np.linalg.norm(
            local_offsets,
            axis=1,
        )
        <= radius_m + 1e-9
    ]

    # remove zero displacement
    local_offsets = local_offsets[
        np.linalg.norm(
            local_offsets,
            axis=1,
        )
        > 1e-9
    ]

    if verbose:
        print("=" * 70)
        print("OPTUNA PROMISING-TURBINE REFINEMENT")
        print("=" * 70)
        print(f"Turbines:               {n_turbines}")
        print(f"Step:                   {step_m} m")
        print(f"Radius:                 {radius_m} m")
        print(
            f"Local grid positions:   "
            f"{len(local_offsets)}"
        )
        print(
            f"Promising turbines:     "
            f"{n_promising_turbines}"
        )
        print(
            f"Trials / turbine:       "
            f"{n_trials_per_turbine}"
        )
        print(
            f"Initial AEP:            "
            f"{current_aep:.6f}"
        )

    # ==================================================
    # OUTER LOOP
    # ==================================================

    for outer_iteration in range(
        max_outer_iterations
    ):

        if verbose:
            print()
            print("=" * 70)
            print(
                f"OUTER ITERATION "
                f"{outer_iteration + 1}/"
                f"{max_outer_iterations}"
            )
            print("=" * 70)
            print(
                f"Current AEP: "
                f"{current_aep:.6f}"
            )

        # ==================================================
        # 1. RANK TURBINES BY WAKE INTERACTION
        #
        # High score = turbine involved in large
        # pairwise wake losses.
        # ==================================================

        wake_scores = current_L.sum(
            axis=1
        )

        promising_turbines = np.argsort(
            wake_scores
        )[::-1][
            :min(
                n_promising_turbines,
                n_turbines,
            )
        ]

        if verbose:
            print()
            print("Most promising turbines:")

            for rank, i in enumerate(
                promising_turbines,
                start=1,
            ):
                print(
                    f"  {rank:02d}. "
                    f"T{i:02d} | "
                    f"wake score="
                    f"{wake_scores[i]:.6f}"
                )

        # ==================================================
        # 2. OPTIMIZE EACH PROMISING TURBINE
        # ==================================================

        turbine_results = []

        for turbine_rank, turbine_idx in enumerate(
            promising_turbines,
            start=1,
        ):

            turbine_idx = int(
                turbine_idx
            )

            origin = current_positions[
                turbine_idx
            ].copy()

            others = np.delete(
                current_positions,
                turbine_idx,
                axis=0,
            )

            # ------------------------------------------
            # Generate local candidates
            # ------------------------------------------

            local_candidates = (
                origin
                + local_offsets
            )

            # boundary
            boundary_mask = np.array(
                [
                    farm_area.covers(
                        Point(x, y)
                    )
                    for x, y
                    in local_candidates
                ]
            )

            local_candidates = (
                local_candidates[
                    boundary_mask
                ]
            )

            if len(local_candidates) == 0:
                continue

            # ------------------------------------------
            # spacing
            # ------------------------------------------

            feasible_candidates = []

            for candidate in local_candidates:

                distances_to_others = (
                    np.linalg.norm(
                        others - candidate,
                        axis=1,
                    )
                )

                if np.all(
                    distances_to_others
                    >= min_distance
                ):
                    feasible_candidates.append(
                        candidate
                    )

            if not feasible_candidates:
                continue

            feasible_candidates = np.asarray(
                feasible_candidates,
                dtype=float,
            )

            feasible_candidates = np.unique(
                np.round(
                    feasible_candidates,
                    6,
                ),
                axis=0,
            )

            feasible_offsets = (
                feasible_candidates
                - origin
            )

            feasible_tree = cKDTree(
                feasible_offsets
            )

            if verbose:
                print()
                print(
                    f"Turbine {turbine_idx:02d} "
                    f"({turbine_rank}/"
                    f"{len(promising_turbines)})"
                )

                print(
                    f"  Wake score:      "
                    f"{wake_scores[turbine_idx]:.6f}"
                )

                print(
                    f"  Feasible moves:  "
                    f"{len(feasible_candidates)}"
                )

            # ==========================================
            # OPTUNA STUDY FOR ONE FIXED TURBINE
            # ==========================================

            cache = {}

            def objective(trial):

                # --------------------------------------
                # TPE proposes a location in local
                # dx/dy space
                # --------------------------------------

                proposed_dx = trial.suggest_float(
                    "dx",
                    -radius_m,
                    radius_m,
                    step=step_m,
                )

                proposed_dy = trial.suggest_float(
                    "dy",
                    -radius_m,
                    radius_m,
                    step=step_m,
                )

                proposed_offset = np.array(
                    [
                        proposed_dx,
                        proposed_dy,
                    ]
                )

                # --------------------------------------
                # Snap to closest FEASIBLE candidate
                #
                # This avoids TrialPruned while
                # preserving spatial structure.
                # --------------------------------------

                _, candidate_idx = (
                    feasible_tree.query(
                        proposed_offset,
                        k=1,
                    )
                )

                candidate_idx = int(
                    candidate_idx
                )

                candidate = (
                    feasible_candidates[
                        candidate_idx
                    ]
                )

                actual_offset = (
                    candidate
                    - origin
                )

                dx = float(
                    actual_offset[0]
                )

                dy = float(
                    actual_offset[1]
                )

                move_distance = float(
                    np.linalg.norm(
                        actual_offset
                    )
                )

                # --------------------------------------
                # Cache by actual candidate
                # --------------------------------------

                key = (
                    round(
                        float(candidate[0]),
                        6,
                    ),
                    round(
                        float(candidate[1]),
                        6,
                    ),
                )

                if key in cache:

                    aep = cache[key]

                else:

                    trial_positions = (
                        current_positions.copy()
                    )

                    trial_positions[
                        turbine_idx
                    ] = candidate

                    aep = float(
                        compute_aep_from_coords(
                            trial_positions
                        )
                    )

                    cache[key] = aep

                delta_aep = (
                    aep - current_aep
                )

                # --------------------------------------
                # Trial metadata
                # --------------------------------------

                trial.set_user_attr(
                    "turbine_idx",
                    turbine_idx,
                )

                trial.set_user_attr(
                    "candidate_idx",
                    candidate_idx,
                )

                trial.set_user_attr(
                    "actual_dx",
                    dx,
                )

                trial.set_user_attr(
                    "actual_dy",
                    dy,
                )

                trial.set_user_attr(
                    "distance_m",
                    move_distance,
                )

                trial.set_user_attr(
                    "candidate_x",
                    float(candidate[0]),
                )

                trial.set_user_attr(
                    "candidate_y",
                    float(candidate[1]),
                )

                trial.set_user_attr(
                    "delta_aep",
                    float(delta_aep),
                )

                if verbose:
                    print(
                        f"    Trial "
                        f"{trial.number + 1:03d} | "
                        f"dx={dx:+6.1f} "
                        f"dy={dy:+6.1f} | "
                        f"AEP={aep:.6f} | "
                        f"ΔAEP="
                        f"{delta_aep:+.6f}"
                    )

                return aep

            sampler = (
                optuna.samplers.TPESampler(
                    seed=(
                        seed
                        + outer_iteration * 1000
                        + turbine_idx
                    ),
                    multivariate=True,
                )
            )

            study = optuna.create_study(
                direction="maximize",
                sampler=sampler,
            )

            study.optimize(
                objective,
                n_trials=n_trials_per_turbine,
                show_progress_bar=False,
            )

            # ------------------------------------------
            # Store trial dataframe
            # ------------------------------------------

            df_trials = (
                study.trials_dataframe()
            )

            df_trials.insert(
                0,
                "outer_iteration",
                outer_iteration + 1,
            )

            df_trials.insert(
                1,
                "turbine_idx",
                turbine_idx,
            )

            all_trials.append(
                df_trials
            )

            # ------------------------------------------
            # Best move for this turbine
            # ------------------------------------------

            best_trial = study.best_trial

            best_candidate = np.array(
                [
                    best_trial.user_attrs[
                        "candidate_x"
                    ],
                    best_trial.user_attrs[
                        "candidate_y"
                    ],
                ],
                dtype=float,
            )

            turbine_best_aep = float(
                study.best_value
            )

            turbine_delta_aep = (
                turbine_best_aep
                - current_aep
            )

            turbine_results.append(
                {
                    "turbine_idx":
                        turbine_idx,

                    "wake_score":
                        float(
                            wake_scores[
                                turbine_idx
                            ]
                        ),

                    "from_position":
                        origin,

                    "to_position":
                        best_candidate,

                    "dx":
                        float(
                            best_candidate[0]
                            - origin[0]
                        ),

                    "dy":
                        float(
                            best_candidate[1]
                            - origin[1]
                        ),

                    "aep":
                        turbine_best_aep,

                    "delta_aep":
                        float(
                            turbine_delta_aep
                        ),

                    "n_feasible_moves":
                        len(
                            feasible_candidates
                        ),

                    "n_unique_aep_evaluations":
                        len(cache),
                }
            )

            if verbose:
                print(
                    f"  Best ΔAEP for T"
                    f"{turbine_idx:02d}: "
                    f"{turbine_delta_aep:+.6f}"
                )

        # ==================================================
        # 3. GLOBAL BEST MOVE
        # ==================================================

        if not turbine_results:

            if verbose:
                print(
                    "\nNo feasible turbine moves."
                )

            break

        best_move = max(
            turbine_results,
            key=lambda x: x["aep"],
        )

        best_aep = best_move[
            "aep"
        ]

        improvement = (
            best_aep
            - current_aep
        )

        if verbose:
            print()
            print("-" * 70)
            print("GLOBAL BEST MOVE")
            print("-" * 70)

            print(
                f"Turbine:       "
                f"{best_move['turbine_idx']}"
            )

            print(
                f"Wake score:    "
                f"{best_move['wake_score']:.6f}"
            )

            print(
                f"From:          "
                f"{best_move['from_position']}"
            )

            print(
                f"To:            "
                f"{best_move['to_position']}"
            )

            print(
                f"dx:            "
                f"{best_move['dx']:+.3f} m"
            )

            print(
                f"dy:            "
                f"{best_move['dy']:+.3f} m"
            )

            print(
                f"AEP before:    "
                f"{current_aep:.6f}"
            )

            print(
                f"AEP after:     "
                f"{best_aep:.6f}"
            )

            print(
                f"ΔAEP:          "
                f"{improvement:+.6f}"
            )

        # ==================================================
        # 4. CONVERGENCE
        # ==================================================

        if improvement <= min_improvement:

            if verbose:
                print(
                    "\nNo improving move found. "
                    "Refinement converged."
                )

            break

        # ==================================================
        # 5. ACCEPT ONLY THE GLOBAL BEST MOVE
        # ==================================================

        moved_idx = int(
            best_move[
                "turbine_idx"
            ]
        )

        old_position = (
            current_positions[
                moved_idx
            ].copy()
        )

        new_position = (
            best_move[
                "to_position"
            ].copy()
        )

        current_positions[
            moved_idx
        ] = new_position

        # ==================================================
        # 6. UPDATE ONLY MOVED TURBINE'S
        #    PAIRWISE WAKE INTERACTIONS
        # ==================================================

        current_L[
            moved_idx,
            :
        ] = 0.0

        current_L[
            :,
            moved_idx
        ] = 0.0

        for j in range(
            n_turbines
        ):

            if j == moved_idx:
                continue

            pair_positions = np.asarray(
                [
                    new_position,
                    current_positions[j],
                ]
            )

            pair_aep = float(
                compute_aep_from_coords(
                    pair_positions
                )
            )

            pair_loss = max(
                0.0,
                2.0 * A0
                - pair_aep,
            )

            current_L[
                moved_idx,
                j
            ] = pair_loss

            current_L[
                j,
                moved_idx
            ] = pair_loss

        # ==================================================
        # HISTORY
        # ==================================================

        history.append(
            {
                "outer_iteration":
                    outer_iteration + 1,

                "turbine_idx":
                    moved_idx,

                "wake_score":
                    best_move[
                        "wake_score"
                    ],

                "from_x":
                    float(
                        old_position[0]
                    ),

                "from_y":
                    float(
                        old_position[1]
                    ),

                "to_x":
                    float(
                        new_position[0]
                    ),

                "to_y":
                    float(
                        new_position[1]
                    ),

                "dx":
                    best_move["dx"],

                "dy":
                    best_move["dy"],

                "aep_before":
                    float(
                        current_aep
                    ),

                "aep_after":
                    float(
                        best_aep
                    ),

                "delta_aep":
                    float(
                        improvement
                    ),
            }
        )

        current_aep = best_aep

    # ==================================================
    # OUTPUT
    # ==================================================

    history_df = pd.DataFrame(
        history
    )

    if all_trials:

        trials_df = pd.concat(
            all_trials,
            ignore_index=True,
        )

    else:

        trials_df = pd.DataFrame()

    if verbose:
        print()
        print("=" * 70)
        print("REFINEMENT COMPLETED")
        print("=" * 70)

        print(
            f"Initial AEP:    "
            f"{initial_aep:.6f}"
        )

        print(
            f"Final AEP:      "
            f"{current_aep:.6f}"
        )

        print(
            f"Improvement:    "
            f"{current_aep - initial_aep:+.6f}"
        )

        print(
            f"Accepted moves: "
            f"{len(history)}"
        )

    return {
        "selected_locations":
            current_positions,

        "initial_aep":
            float(initial_aep),

        "final_aep":
            float(current_aep),

        "improvement_aep":
            float(
                current_aep
                - initial_aep
            ),

        "n_swaps":
            len(history),

        "wake_interaction_matrix":
            current_L,

        "history_df":
            history_df,

        "trials_df":
            trials_df,
    }


def build_local_candidates(
    gurobi_positions,
    farm_area,
    radius_m,
    step_m,
    min_distance=396.0,
):
    gurobi_positions = np.asarray(gurobi_positions, dtype=float)

    # offset locali
    offsets_1d = np.arange(
        -radius_m,
        radius_m + step_m * 0.5,
        step_m,
    )

    dx, dy = np.meshgrid(
        offsets_1d,
        offsets_1d,
        indexing="xy",
    )

    offsets = np.column_stack([
        dx.ravel(),
        dy.ravel(),
    ])

    # tengo solo punti dentro il raggio
    offsets = offsets[
        np.linalg.norm(offsets, axis=1) <= radius_m
    ]

    # genero candidati attorno a ogni turbina Gurobi
    candidate_locations = (
        gurobi_positions[:, None, :]
        + offsets[None, :, :]
    ).reshape(-1, 2)

    # aggiungo esplicitamente le posizioni Gurobi originali
    candidate_locations = np.vstack([
        candidate_locations,
        gurobi_positions,
    ])

    # tengo solo punti dentro il farm boundary
    mask = np.array([
        farm_area.covers(Point(x, y))
        for x, y in candidate_locations
    ])

    candidate_locations = candidate_locations[mask]

    # rimuovo duplicati
    candidate_locations = np.unique(
        np.round(candidate_locations, 6),
        axis=0,
    )

    # invalid pairs
    tree = cKDTree(candidate_locations)

    invalid_pairs = tree.query_pairs(
        r=min_distance,
        output_type="ndarray",
    )

    # query_pairs include anche distanza == min_distance:
    # noi vogliamo solo < min_distance
    if len(invalid_pairs) > 0:
        d = np.linalg.norm(
            candidate_locations[invalid_pairs[:, 0]]
            - candidate_locations[invalid_pairs[:, 1]],
            axis=1,
        )

        invalid_pairs = invalid_pairs[
            d < min_distance
        ]

    return (
        candidate_locations,
        invalid_pairs.astype(np.int32),
    )

In [5]:
WAKE_LOSS_PATH = f"../../results/precomputed/gurobi/wake_loss_gurobi_res_4m_rad_8.npz"

wake_loss_matrix = np.load(WAKE_LOSS_PATH)["wake_loss_matrix"]

In [6]:
STARTING_LAYOUT = "../../results/layouts/gurobi/gurobi_local_4m_1800s_warmstart.yaml"
gurobi_positions = load_layout_coordinates(STARTING_LAYOUT)

_, farm_area = get_farm_area()

candidate_locations, invalid_pairs = build_local_candidates(
    gurobi_positions=gurobi_positions,
    farm_area=farm_area,
    radius_m=8,
    step_m=4,
)


result = optuna_promising_turbines_refinement(
    layout_yaml=STARTING_LAYOUT,
    candidate_locations=candidate_locations,
    wake_loss_matrix=wake_loss_matrix,
    step_m=2,
    radius_m=16,
    n_promising_turbines=15,
    n_trials_per_turbine=30,
    max_outer_iterations=20,
)

[I 2026-09-30 18:09:04,411] A new study created in memory with name: no-name-3acadd85-dc35-48d8-adb9-1be82b96b068


OPTUNA PROMISING-TURBINE REFINEMENT
Turbines:               81
Step:                   2 m
Radius:                 16 m
Local grid positions:   196
Promising turbines:     15
Trials / turbine:       30
Initial AEP:            2901480.912479

OUTER ITERATION 1/20
Current AEP: 2901480.912479

Most promising turbines:
  01. T43 | wake score=66373.851296
  02. T42 | wake score=66364.864978
  03. T57 | wake score=65682.678261
  04. T58 | wake score=65671.138588
  05. T39 | wake score=65573.359723
  06. T38 | wake score=65571.963265
  07. T32 | wake score=65466.572192
  08. T31 | wake score=65443.604234
  09. T29 | wake score=65147.393532
  10. T28 | wake score=65143.040965
  11. T61 | wake score=65130.197859
  12. T62 | wake score=65119.544948
  13. T11 | wake score=63441.446106
  14. T10 | wake score=63423.676100
  15. T72 | wake score=60998.448529

Turbine 43 (1/15)
  Wake score:      66373.851296
  Feasible moves:  196


[I 2026-09-30 18:09:20,654] Trial 0 finished with value: 2901479.967747501 and parameters: {'dx': 4.0, 'dy': 0.0}. Best is trial 0 with value: 2901479.967747501.


    Trial 001 | dx=  +4.0 dy=  +0.0 | AEP=2901479.967748 | ΔAEP=-0.944732


[I 2026-09-30 18:09:37,798] Trial 1 finished with value: 2901477.116496506 and parameters: {'dx': -6.0, 'dy': 6.0}. Best is trial 0 with value: 2901479.967747501.


...

    Trial 001 | dx=  -2.0 dy=  +2.0 | AEP=2901477.833063 | ΔAEP=-3.079416


[I 2026-09-30 18:44:14,934] Trial 1 finished with value: 2901489.507021812 and parameters: {'dx': 12.0, 'dy': -10.0}. Best is trial 1 with value: 2901489.507021812.


    Trial 002 | dx= +12.0 dy= -10.0 | AEP=2901489.507022 | ΔAEP=+8.594543


[W 2026-09-30 18:44:30,014] Trial 2 failed with parameters: {'dx': 14.0, 'dy': 10.0} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/maicolnicolini/code/qubo-wind-farm-layout/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_113098/3918287723.py", line 473, in objective
    compute_aep_from_coords(
  File "/home/maicolnicolini/code/qubo-wind-farm-layout/notebooks/gurobi/../../src/qubo_windfarm_layout/model.py", line 336, in compute_aep_from_coords
    aep_by_direction = calcAEPcs3(
                       ^^^^^^^^^^^
  File "/home/maicolnicolini/code/qubo-wind-farm-layout/notebooks/gurobi/../../external/thomas-wflo-benchmark/src/physics-models/provided_model.py", line 118, in calcAEPcs3
    dir_loss = GaussianWake(frame_coords, turb_diam)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mai

KeyboardInterrupt: 